# NeSLE: verify every claimed number

Re-measures every performance claim in the README from a clean clone and prints
a claimed-vs-measured PASS / FAIL table.

All logic is in `benchmarks/verify_claims.py`; this notebook supplies a ROM and
runs it. That script reuses the functions in `benchmarks/gpu_vs_cpu.py`, so the
protocol matches the published tables exactly (frameskip 4, RIGHT held,
30 warmup + 200 timed steps, `render_frame=False, copy_obs=False`).

**Runtime:** Runtime > Change runtime type > **A100 GPU**.

**Supply a legally obtained `Super Mario Bros. (World).nes`** (iNES, mapper 0).
No ROM ships with the repository or this notebook.

Results are written to `/content/verification.json` **after every stage**, and
printed in full at the end, so an interrupted runtime cannot lose them.

Total time: roughly 20 to 30 minutes, most of it the nes-py baselines.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code

In [ ]:
!rm -rf /content/NeSLE
!git clone --depth 1 -b docs/verified-numbers https://github.com/hbofz/NeSLE.git /content/NeSLE

## 3. ROM

Upload the file, or point `DRIVE_ROM` at a mounted Drive path.

In [ ]:
from pathlib import Path
import hashlib

ROM = Path("/content/rom.nes")
DRIVE_ROM = ""  # e.g. "/content/drive/MyDrive/mario_rl/roms/Super Mario Bros. (World).nes"

if DRIVE_ROM:
    import shutil; shutil.copy(DRIVE_ROM, ROM)
elif not ROM.exists():
    from google.colab import files
    up = files.upload()
    ROM.write_bytes(up[next(iter(up))])

d = ROM.read_bytes()
assert d[:4] == b"NES\x1a", "not an iNES ROM"
mapper = (d[6] >> 4) | (d[7] & 0xF0)
assert mapper == 0, f"NeSLE supports mapper 0 only, got {mapper}"
print(f"{len(d):,} bytes  mapper {mapper}  sha1={hashlib.sha1(d).hexdigest()}")

## 4. Run everything

Installs the package, builds the CUDA extension, runs the 69 tests and the
three falsifiability checks, sweeps throughput to 131,072 envs, measures device
memory with the batch resident, and benchmarks nes-py on one core and on all
cores.

Output streams live. Exits non-zero if any claim, test, or check fails.

In [ ]:
!python /content/NeSLE/benchmarks/verify_claims.py --repo /content/NeSLE --rom /content/rom.nes --out /content/verification.json

## 5. Keep the evidence

In [ ]:
import json
r = json.load(open("/content/verification.json"))

for v in r.get("verdicts", []):
    m = f"{v['measured']:,.0f}" if v["measured"] else "-"
    print(f"{v['envs']:>9,} envs   claimed {v['claimed']:>12,}   measured {m:>12}   {v['verdict']}")

print("\ncrossover:", r.get("crossover_envs"), "envs")
print("tests:", r["tests"]["summary"])
print("nes-py single:", r.get("nespy_single"))
print("nes-py multicore:", r.get("nespy_multicore"))

from google.colab import files
files.download("/content/verification.json")


If the download is blocked, copy the JSON printed at the end of the previous
cell. Commit it to `docs/data/`.